# 01 · Data Exploration
Multilingual RAG Evaluation & LLM Hallucination Detection System

This notebook explores the questions dataset and multilingual knowledge base built in Phase 2 (`src/build_seed_data.py`).

In [ ]:
import sys
from pathlib import Path

# Notebooks run from notebooks/, but the src package lives at the project
# root -- add the project root to sys.path so `from src import ...` works.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 100)


In [ ]:
from src import data_loader

questions_df = data_loader.load_questions()
kb_df = data_loader.load_knowledge_base()

print(f'Questions: {len(questions_df)}')
print(f'Knowledge base documents: {len(kb_df)}')

## Dataset design
This project uses **15 underlying facts** (5 Geography, 5 History, 5 Science), each asked in all 3 languages -> **45 questions total**. Every question's `question_id` (e.g. `GEO01_en`) encodes its `fact_id` (`GEO01`) and language code (`en`), which lets us compare retrieval quality and accuracy for the *same fact* across languages -- a stronger design than comparing unrelated per-language question sets.

In [ ]:
print('Questions per language:')
display(questions_df['language'].value_counts())

print('\nQuestions per category:')
display(questions_df['category'].value_counts())

In [ ]:
questions_df.sample(6, random_state=42)

## Knowledge base
The knowledge base contains one matching document per fact per language (45 docs) plus **9 distractor documents** (topically related but not answering any question) that make retrieval non-trivial instead of a guaranteed top-1 match.

In [ ]:
print('KB documents per language:')
display(kb_df['language'].value_counts())

print(f"\nDistractor documents: {int(kb_df['is_distractor'].sum())}")
kb_df[kb_df['is_distractor']][['document_id', 'category', 'language', 'content']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
questions_df['language'].value_counts().plot(kind='bar', ax=axes[0], color='#0891b2', title='Questions by Language')
questions_df['category'].value_counts().plot(kind='bar', ax=axes[1], color='#f59e0b', title='Questions by Category')
for ax in axes:
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Notes
- All knowledge base content is **original reference material written for this project** (see `source` column) -- it does not claim to be scraped from any external publication.
- Facts were chosen to be simple, verifiable, and unambiguous, making this a controlled evaluation environment rather than an open-domain benchmark.